# Landscape → Painting with CycleGAN (unpaired image-to-image translation)

This notebook trains a **CycleGAN** that turns landscape photographs into paintings in a chosen art style (default: **Naive Art / Primitivism**). There are **no paired examples**. The model learns two mappings, *G: photo → painting* and *F: painting → photo*, and keeps scene content with a **cycle-consistency** loss.

**Pipeline:** data discovery → augmentation → ResNet-9 generators + 70×70 PatchGAN discriminators → LSGAN + cycle + identity losses → mixed-precision training with checkpoint/resume → qualitative gallery → optional FID score → inference on your own photos.

> **Running on Kaggle:** add the two datasets in *Input*, turn on **GPU** and **Internet**, then use *Save Version → Save & Run All (Commit)*. Step-by-step instructions are in the repo README.

## 1. Configuration
Change values only in this cell. To try a different style, set `ART_STYLE` to any WikiArt folder name, for example `Cubism`, `Pop_Art`, `Ukiyo_e`, `Northern_Renaissance`, `Post_Impressionism` or `Pointillism`.

In [ ]:
import os
from pathlib import Path

# ---- Data -------------------------------------------------------------------
ART_STYLE      = "Naive_Art_Primitivism"   # WikiArt style folder name
LANDSCAPE_DIR  = None   # None = auto-discover under /kaggle/input (folder name contains "landscape")
ART_DIR        = None   # None = auto-discover a folder named ART_STYLE under /kaggle/input
N_TEST         = 64     # landscapes held out from training (used for galleries / FID)
DROP_GRAYSCALE = True   # the landscape dataset contains black-and-white photos; they teach no colour style

# ---- Model / training (CycleGAN paper defaults, tuned identity weight) -------
IMG_SIZE        = 256
LOAD_SIZE       = 286   # resize to this, then random-crop IMG_SIZE (paper augmentation)
BATCH_SIZE      = 4
EPOCHS          = 50
DECAY_START     = 25    # LR is constant until this epoch, then decays linearly to 0
LR              = 2e-4
BETAS           = (0.5, 0.999)
LAMBDA_CYCLE    = 10.0
LAMBDA_IDENTITY = 0.5   # lambda_id = 5 gave near-identity outputs; 0.5 balanced style and content best
POOL_SIZE       = 50    # history buffer of generated images for the discriminators
N_RES_BLOCKS    = 9
USE_AMP         = True  # mixed precision: ~1.6-2x faster on T4
SEED            = 42

# ---- Kaggle session safety ---------------------------------------------------
TIME_BUDGET_HOURS = 11.3  # Kaggle stops sessions at 12 h; training stops cleanly before that
SAVE_EVERY        = 10    # also keep a generator-only snapshot every N epochs
TRAIN             = True    # False = skip training, just evaluate the latest checkpoint (see README)
RESUME_FROM       = "auto"  # "auto" = latest checkpoints/last.pt under /kaggle/working or /kaggle/input; or a path; or None
COMPUTE_FID       = True    # Inception-v3 weights (~100 MB) are downloaded once at startup

# ---- Paths -------------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")

# Smoke-test mode (used by CI): tiny synthetic data, runs on CPU in about a minute.
SMOKE_TEST = os.environ.get("L2A_SMOKE_TEST") == "1"
if SMOKE_TEST:
    IMG_SIZE, LOAD_SIZE, BATCH_SIZE, EPOCHS, DECAY_START = 64, 72, 2, 2, 1
    N_TEST, N_RES_BLOCKS, POOL_SIZE = 4, 2, 4
    COMPUTE_FID = os.environ.get("L2A_SMOKE_FID") == "1"
    WORK_DIR = Path(os.environ.get("L2A_WORK_DIR", "./smoke_outputs"))

CKPT_DIR, SAMPLE_DIR, RESULT_DIR = WORK_DIR / "checkpoints", WORK_DIR / "samples", WORK_DIR / "results"
for d in (CKPT_DIR, SAMPLE_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 2. Setup

In [ ]:
import csv, json, math, random, time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True  # some WikiArt files are truncated

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu")
AMP = USE_AMP and device.type == "cuda"
torch.backends.cudnn.benchmark = device.type == "cuda"
NUM_WORKERS = 0 if SMOKE_TEST else min(4, os.cpu_count() or 1)

print(f"torch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else "")
      + f" | AMP: {AMP} | workers: {NUM_WORKERS}")
if device.type != "cuda" and not SMOKE_TEST:
    print("WARNING: no GPU found. On Kaggle: Settings -> Accelerator -> GPU T4 x2.")

### Internet and FID weights check
FID is computed with **torchvision's Inception-v3 and scipy**, both preinstalled on Kaggle, so **no `pip install` is needed**. The weights (~100 MB) are downloaded **here, before training**, so a network problem shows up in the first minute instead of after hours of training. Without network access, the notebook looks for `inception_v3_google-0cc3c7bd.pth` in any attached dataset (see README).

In [ ]:
import socket
from torchvision.models import inception_v3, Inception_V3_Weights

def has_internet(host="download.pytorch.org"):
    try:
        socket.getaddrinfo(host, 443)
        return True
    except OSError:
        return False

FID_WEIGHTS_FILE = Path(Inception_V3_Weights.IMAGENET1K_V1.url).name  # inception_v3_google-0cc3c7bd.pth

def load_fid_inception():
    # ImageNet Inception-v3 returning 2048-d pool features; input is RGB in [-1, 1] (our normalisation).
    net = inception_v3(weights=None, aux_logits=True, transform_input=False, init_weights=False)
    cached = Path(torch.hub.get_dir()) / "checkpoints" / FID_WEIGHTS_FILE
    if cached.exists() or has_internet():
        state = Inception_V3_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
    else:
        local = next(INPUT_ROOT.glob(f"**/{FID_WEIGHTS_FILE}"), None) if INPUT_ROOT.exists() else None
        if local is None:
            raise RuntimeError(f"no network and {FID_WEIGHTS_FILE} not found under {INPUT_ROOT}")
        state = torch.load(local, map_location="cpu")
    net.load_state_dict(state)
    net.fc = nn.Identity()
    return net.eval()

print("Internet:", "OK" if has_internet() else "NOT reachable (check Settings -> Internet, then restart the session)")
FID_NET = None
if COMPUTE_FID:
    try:
        FID_NET = load_fid_inception()
        print("FID: Inception-v3 weights ready.")
    except Exception as e:
        print(f"WARNING: FID will be skipped ({e}). Training is unaffected.")

## 3. Data
* **Domain X (photos):** [Landscape Pictures](https://www.kaggle.com/datasets/arnaud58/landscape-pictures), about 4.3k images
* **Domain Y (paintings):** one style folder of [WikiArt](https://www.kaggle.com/datasets/steubk/wikiart)

The folders are found automatically under `/kaggle/input`, so the notebook works whatever path Kaggle mounts the datasets at. Unreadable files are skipped, not allowed to crash training. A fixed set of `N_TEST` landscapes is held out and never used for training.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXTS)

def find_dir(root, predicate, max_depth=5):
    # Breadth-first search for the shallowest directory that matches and contains images.
    root = Path(root)
    if not root.exists():
        return None
    frontier = [root]
    for _ in range(max_depth):
        nxt = []
        for d in frontier:
            try:
                subdirs = sorted(p for p in d.iterdir() if p.is_dir())
            except PermissionError:
                continue
            for s in subdirs:
                if predicate(s) and any(p.suffix.lower() in IMG_EXTS for p in s.iterdir()):
                    return s
            nxt.extend(subdirs)
        frontier = nxt
    return None

def make_synthetic_domain(folder, n, kind):
    # Smoke test only: random images so the pipeline runs without the real datasets.
    folder.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0 if kind == "photo" else 1)
    for i in range(n):
        arr = rng.integers(0, 255, (80, 96, 3), dtype=np.uint8)
        if kind == "photo" and i < 2:  # a couple of black-and-white photos to exercise DROP_GRAYSCALE
            arr[:] = arr[..., :1]
        Image.fromarray(arr).save(folder / f"{kind}_{i}.jpg")
    return folder

if SMOKE_TEST:
    LANDSCAPE_DIR = make_synthetic_domain(WORK_DIR / "data" / "landscapes", 12, "photo")
    ART_DIR = make_synthetic_domain(WORK_DIR / "data" / ART_STYLE, 8, "art")
else:
    LANDSCAPE_DIR = Path(LANDSCAPE_DIR) if LANDSCAPE_DIR else find_dir(INPUT_ROOT, lambda p: "landscape" in p.name.lower())
    ART_DIR = Path(ART_DIR) if ART_DIR else find_dir(INPUT_ROOT, lambda p: p.name.lower() == ART_STYLE.lower())

assert LANDSCAPE_DIR and LANDSCAPE_DIR.exists(), "Landscape folder not found. Add the 'landscape-pictures' dataset or set LANDSCAPE_DIR."
assert ART_DIR and ART_DIR.exists(), f"Art folder '{ART_STYLE}' not found. Add the 'wikiart' dataset or set ART_DIR."

photo_files = list_images(LANDSCAPE_DIR)
art_files = list_images(ART_DIR)
random.Random(SEED).shuffle(photo_files)
test_photo_files, train_photo_files = photo_files[:N_TEST], photo_files[N_TEST:]

def is_grayscale(path, tol=8.0):
    # True if the colour channels are (almost) identical, i.e. a black-and-white photo.
    try:
        im = Image.open(path)
        im.draft("RGB", (64, 64))  # fast JPEG decode at low resolution
        a = np.asarray(im.convert("RGB").resize((32, 32)), dtype=np.int16)
        return float((a.max(-1) - a.min(-1)).mean()) < tol
    except Exception:
        return False

# Filter AFTER the split so the held-out test set stays disjoint from earlier training runs (resume-safe).
n_gray_removed = 0
if DROP_GRAYSCALE:
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex:
        gray = dict(zip(photo_files, ex.map(is_grayscale, photo_files)))
    n_gray_removed = sum(gray.values())
    test_photo_files = [f for f in test_photo_files if not gray[f]]
    train_photo_files = [f for f in train_photo_files if not gray[f]]
    print(f"Removed {n_gray_removed} black-and-white photos ({100 * n_gray_removed / len(photo_files):.1f}%)")

print(f"Photos : {LANDSCAPE_DIR} -> {len(train_photo_files)} train / {len(test_photo_files)} test")
print(f"Art    : {ART_DIR} -> {len(art_files)} images")
assert len(train_photo_files) >= BATCH_SIZE and len(art_files) >= BATCH_SIZE, "Not enough images."

In [ ]:
def safe_open(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:  # corrupt / unreadable file
        print(f"skip {path}: {e}")
        return None

normalize = T.Normalize((0.5,) * 3, (0.5,) * 3)  # [0,1] -> [-1,1], matches tanh output
train_tf = T.Compose([
    T.Resize((LOAD_SIZE, LOAD_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.ToTensor(), normalize,
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(), normalize,
])

def denorm(x):
    return (x.detach().float().cpu() * 0.5 + 0.5).clamp(0, 1)

class UnpairedDataset(Dataset):
    # Each item is a photo plus a *random* painting. The domains are unaligned, as in the CycleGAN paper.
    def __init__(self, files_x, files_y, transform):
        self.fx, self.fy, self.tf = files_x, files_y, transform
    def __len__(self):
        return len(self.fx)
    def _load(self, files, idx):
        for _ in range(10):  # retry with another random file if one is corrupt
            img = safe_open(files[idx])
            if img is not None:
                return self.tf(img)
            idx = random.randrange(len(files))
        raise RuntimeError("Too many unreadable images in a row.")
    def __getitem__(self, i):
        return self._load(self.fx, i), self._load(self.fy, random.randrange(len(self.fy)))

class ImageFolderList(Dataset):
    def __init__(self, files, transform):
        self.files, self.tf = files, transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = safe_open(self.files[i]) or Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        return self.tf(img)

train_loader = DataLoader(
    UnpairedDataset(train_photo_files, art_files, train_tf), batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=device.type == "cuda", drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
test_photos = torch.stack([ImageFolderList(test_photo_files, eval_tf)[i] for i in range(min(4, len(test_photo_files)))])
print(f"{len(train_loader)} iterations / epoch")

In [ ]:
def show_row(images, title, n=4):
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.4))
    for ax, img in zip(np.atleast_1d(axes), images[:n]):
        ax.imshow(denorm(img).permute(1, 2, 0)); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

xb, yb = next(iter(train_loader))
show_row(xb, "Domain X: landscape photos (augmented)", n=min(4, len(xb)))
show_row(yb, f"Domain Y: {ART_STYLE.replace('_', ' ')}", n=min(4, len(yb)))

## 4. Model
* **Generator:** ResNet-9 (Johnson et al.). c7s1-64 → d128 → d256 → 9× residual blocks → u128 → u64 → c7s1-3 → tanh. It uses reflection padding to avoid border artifacts, and the residual blocks let the network keep scene structure while it changes texture and colour.
* **Discriminator:** 70×70 **PatchGAN**: C64 → C128 → C256 → C512 → 1-channel map. It judges whether each local patch is real, which fits brush-stroke texture well.
* Weights are initialised with N(0, 0.02), as in the paper.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch),
        )
    def forward(self, x):
        return x + self.block(x)

class ResNetGenerator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, ngf=64, n_blocks=9):
        super().__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(in_ch, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        ch = ngf
        for _ in range(2):  # downsample
            layers += [nn.Conv2d(ch, ch * 2, 3, 2, 1), nn.InstanceNorm2d(ch * 2), nn.ReLU(True)]
            ch *= 2
        layers += [ResidualBlock(ch) for _ in range(n_blocks)]
        for _ in range(2):  # upsample
            layers += [nn.ConvTranspose2d(ch, ch // 2, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(ch // 2), nn.ReLU(True)]
            ch //= 2
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(ch, out_ch, 7), nn.Tanh()]
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x)

class PatchGANDiscriminator(nn.Module):
    def __init__(self, in_ch=3, ndf=64):
        super().__init__()
        def block(i, o, stride, norm=True):
            return [nn.Conv2d(i, o, 4, stride, 1)] + ([nn.InstanceNorm2d(o)] if norm else []) + [nn.LeakyReLU(0.2, True)]
        self.model = nn.Sequential(
            *block(in_ch, ndf, 2, norm=False), *block(ndf, ndf * 2, 2), *block(ndf * 2, ndf * 4, 2),
            *block(ndf * 4, ndf * 8, 1), nn.Conv2d(ndf * 8, 1, 4, 1, 1),
        )
    def forward(self, x):
        return self.model(x)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

G_XY = ResNetGenerator(n_blocks=N_RES_BLOCKS).to(device)   # photo -> painting
G_YX = ResNetGenerator(n_blocks=N_RES_BLOCKS).to(device)   # painting -> photo
D_X, D_Y = PatchGANDiscriminator().to(device), PatchGANDiscriminator().to(device)
for net in (G_XY, G_YX, D_X, D_Y):
    net.apply(init_weights)

count = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"Generator: {count(G_XY):.2f}M params | Discriminator: {count(D_X):.2f}M params")
with torch.no_grad():  # shape sanity check
    _x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    assert G_XY(_x).shape == _x.shape
    print("PatchGAN output map:", tuple(D_X(_x).shape))

## 5. Losses, optimisers and schedule
$$\mathcal{L} = \mathcal{L}_{GAN}(G, D_Y) + \mathcal{L}_{GAN}(F, D_X) + \lambda_{cyc}\,\big(\lVert F(G(x)) - x\rVert_1 + \lVert G(F(y)) - y\rVert_1\big) + \lambda_{id}\,\big(\lVert G(y) - y\rVert_1 + \lVert F(x) - x\rVert_1\big)$$

* **LSGAN** (MSE) adversarial loss, which trains more stably than BCE.
* An **image pool** of 50 past fakes for the discriminators, so the generator and discriminator are less likely to oscillate.
* **Adam** (lr 2e-4, β = (0.5, 0.999)) with the LR held constant, then decayed **linearly to 0**.

In [ ]:
class ImagePool:
    # Buffer of previously generated images (Shrivastava et al. 2017), used by the CycleGAN discriminators.
    def __init__(self, size):
        self.size, self.images = size, []
    def query(self, batch):
        if self.size == 0:
            return batch
        out = []
        for img in batch.detach():
            img = img.unsqueeze(0)
            if len(self.images) < self.size:
                self.images.append(img); out.append(img)
            elif random.random() < 0.5:
                j = random.randrange(self.size)
                out.append(self.images[j].clone()); self.images[j] = img
            else:
                out.append(img)
        return torch.cat(out)

mse, l1 = nn.MSELoss(), nn.L1Loss()
def gan_loss(pred, is_real):
    pred = pred.float()
    return mse(pred, torch.ones_like(pred) if is_real else torch.zeros_like(pred))

opt_G = torch.optim.Adam(list(G_XY.parameters()) + list(G_YX.parameters()), lr=LR, betas=BETAS)
opt_D = torch.optim.Adam(list(D_X.parameters()) + list(D_Y.parameters()), lr=LR, betas=BETAS)
lr_lambda = lambda e: 1.0 - max(0, e - DECAY_START) / float(EPOCHS - DECAY_START + 1)
sched_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_D = torch.optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=AMP)
pool_X, pool_Y = ImagePool(POOL_SIZE), ImagePool(POOL_SIZE)

def set_requires_grad(nets, flag):
    for n in nets:
        for p in n.parameters():
            p.requires_grad_(flag)

## 6. Checkpointing and resume
`last.pt` holds the full training state (all four networks, optimisers, schedulers, AMP scaler and loss history). It is written to disk **atomically after every epoch**, so a session that is stopped or times out loses at most one epoch.

**To continue training in a new session:** add the output of your previous notebook version as an input (*Add Input → Your Work → Notebook output*). `RESUME_FROM = "auto"` finds it.

In [ ]:
CONFIG = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, tuple, type(None)))}

def save_checkpoint(epoch, history, path=CKPT_DIR / "last.pt"):
    state = {
        "epoch": epoch, "history": history, "config": CONFIG,
        "G_XY": G_XY.state_dict(), "G_YX": G_YX.state_dict(), "D_X": D_X.state_dict(), "D_Y": D_Y.state_dict(),
        "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict(),
        "sched_G": sched_G.state_dict(), "sched_D": sched_D.state_dict(), "scaler": scaler.state_dict(),
    }
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)  # atomic: a crash mid-save never corrupts last.pt

def find_resume_checkpoint():
    if RESUME_FROM in (None, False, ""):
        return None
    if RESUME_FROM != "auto":
        return Path(RESUME_FROM)
    candidates = [CKPT_DIR / "last.pt"]
    if INPUT_ROOT.exists() and not SMOKE_TEST:
        candidates += list(INPUT_ROOT.glob("**/checkpoints/last.pt"))
    best, best_epoch = None, -1
    print("Checkpoint candidates:", [str(c) for c in candidates if c.exists()] or "none found")
    for c in candidates:
        if not c.exists():
            continue
        meta = torch.load(c, map_location="cpu", weights_only=False)
        if meta.get("config", {}).get("ART_STYLE") == ART_STYLE and meta["epoch"] > best_epoch:
            best, best_epoch = c, meta["epoch"]
    return best

start_epoch, history = 0, []
ckpt_path = find_resume_checkpoint()
if ckpt_path:
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    for name, obj in [("G_XY", G_XY), ("G_YX", G_YX), ("D_X", D_X), ("D_Y", D_Y), ("opt_G", opt_G),
                      ("opt_D", opt_D), ("sched_G", sched_G), ("sched_D", sched_D), ("scaler", scaler)]:
        obj.load_state_dict(ck[name])
    start_epoch, history = ck["epoch"], ck["history"]
    print(f"Resumed from {ckpt_path} at epoch {start_epoch}/{EPOCHS}")
else:
    if INPUT_ROOT.exists() and not SMOKE_TEST and any(INPUT_ROOT.glob("notebooks/*/*")):
        print("WARNING: an attached notebook output contains no checkpoints/last.pt. Kaggle mounts an empty folder "
              "when that version failed; use its Output tab -> 'New dataset' and attach the dataset instead (see README).")
    if not TRAIN:
        raise FileNotFoundError("TRAIN=False but no checkpoint was found. Attach the previous run's output "
                                "(or a dataset made from it) or set RESUME_FROM to the path of last.pt.")
    print("Starting training from scratch.")

## 7. Training
At the end of each epoch the notebook saves `last.pt`, appends the mean losses to `history.csv` and writes a fixed-input preview to `samples/`. Previews are saved to files instead of being shown inline, so the notebook stays small. Training stops cleanly before Kaggle's 12-hour limit.

In [ ]:
LOSS_KEYS = ["G_total", "G_adv", "cycle", "identity", "D_X", "D_Y"]

def train_one_epoch(epoch):
    for n in (G_XY, G_YX, D_X, D_Y):
        n.train()
    sums = dict.fromkeys(LOSS_KEYS, 0.0)
    pbar = tqdm(train_loader, desc=f"epoch {epoch + 1}/{EPOCHS}", leave=False)
    for real_x, real_y in pbar:
        real_x, real_y = real_x.to(device, non_blocking=True), real_y.to(device, non_blocking=True)

        # ---- Generators ----
        set_requires_grad([D_X, D_Y], False)
        opt_G.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=AMP):
            fake_y, fake_x = G_XY(real_x), G_YX(real_y)
            rec_x, rec_y = G_YX(fake_y), G_XY(fake_x)
            loss_adv = gan_loss(D_Y(fake_y), True) + gan_loss(D_X(fake_x), True)
            loss_cyc = l1(rec_x.float(), real_x) + l1(rec_y.float(), real_y)
            if LAMBDA_IDENTITY > 0:
                loss_id = l1(G_XY(real_y).float(), real_y) + l1(G_YX(real_x).float(), real_x)
            else:
                loss_id = torch.zeros((), device=device)
            loss_G = loss_adv + LAMBDA_CYCLE * loss_cyc + LAMBDA_IDENTITY * loss_id
        scaler.scale(loss_G).backward()
        scaler.step(opt_G)

        # ---- Discriminators (on a history of fakes) ----
        set_requires_grad([D_X, D_Y], True)
        opt_D.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=AMP):
            loss_DX = 0.5 * (gan_loss(D_X(real_x), True) + gan_loss(D_X(pool_X.query(fake_x.float())), False))
            loss_DY = 0.5 * (gan_loss(D_Y(real_y), True) + gan_loss(D_Y(pool_Y.query(fake_y.float())), False))
        scaler.scale(loss_DX + loss_DY).backward()
        scaler.step(opt_D)
        scaler.update()

        vals = [loss_G, loss_adv, loss_cyc, loss_id, loss_DX, loss_DY]
        if not all(torch.isfinite(v) for v in vals):
            raise FloatingPointError(f"Non-finite loss at epoch {epoch + 1}. Try USE_AMP=False or a lower LR.")
        for k, v in zip(LOSS_KEYS, vals):
            sums[k] += v.item()
        pbar.set_postfix(G=f"{loss_G.item():.2f}", cyc=f"{loss_cyc.item():.3f}", DX=f"{loss_DX.item():.3f}", DY=f"{loss_DY.item():.3f}")
    return {k: v / len(train_loader) for k, v in sums.items()}

@torch.no_grad()
def save_preview(epoch):
    G_XY.eval()
    fake = G_XY(test_photos.to(device)).float().cpu()
    grid = torch.cat([denorm(test_photos), denorm(fake)], dim=3)  # photo | painting, side by side
    save_image(grid, SAMPLE_DIR / f"epoch_{epoch:03d}.jpg", nrow=1, padding=4)

def write_history_csv():
    with open(WORK_DIR / "history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["epoch", "lr", "seconds"] + LOSS_KEYS)
        w.writeheader(); w.writerows(history)

t_start, epoch_times = time.time(), []
if not TRAIN:
    print(f"TRAIN=False: skipping training, evaluating the checkpoint from epoch {start_epoch}.")
for epoch in range(start_epoch, EPOCHS if TRAIN else start_epoch):
    if epoch_times and (time.time() - t_start + 1.15 * max(epoch_times)) / 3600 > TIME_BUDGET_HOURS:
        print(f"Time budget reached. Stopping after epoch {epoch}; re-run with this output as input to resume.")
        break
    t0 = time.time()
    stats = train_one_epoch(epoch)
    lr_now = opt_G.param_groups[0]["lr"]
    sched_G.step(); sched_D.step()
    epoch_times.append(time.time() - t0)
    history.append({"epoch": epoch + 1, "lr": lr_now, "seconds": round(epoch_times[-1], 1), **{k: round(v, 5) for k, v in stats.items()}})

    save_checkpoint(epoch + 1, history)
    write_history_csv()
    save_preview(epoch + 1)
    if (epoch + 1) % SAVE_EVERY == 0 or epoch + 1 == EPOCHS:
        torch.save(G_XY.state_dict(), CKPT_DIR / f"G_photo2art_epoch{epoch + 1:03d}.pt")
    print(f"epoch {epoch + 1:3d}/{EPOCHS} | " + " | ".join(f"{k} {v:.3f}" for k, v in stats.items())
          + f" | lr {lr_now:.1e} | {epoch_times[-1] / 60:.1f} min")

torch.save(G_XY.state_dict(), WORK_DIR / f"generator_{ART_STYLE}.pt")  # final inference weights
print("Done. Final generator:", WORK_DIR / f"generator_{ART_STYLE}.pt")

## 8. Training curves

In [ ]:
if history:
    ep = [h["epoch"] for h in history]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
    a1.plot(ep, [h["G_total"] for h in history], label="generator (total)")
    a1.plot(ep, [h["G_adv"] for h in history], label="generator (adversarial)")
    a1.set_title("Generator"); a1.set_xlabel("epoch"); a1.legend()
    for k in ("cycle", "identity", "D_X", "D_Y"):
        a2.plot(ep, [h[k] for h in history], label=k)
    a2.set_title("Cycle / identity / discriminators"); a2.set_xlabel("epoch"); a2.legend()
    plt.tight_layout(); fig.savefig(WORK_DIR / "losses.png", dpi=120); plt.show()

## 9. Results on held-out photos
GAN losses are not a good measure of image quality. The generator and discriminator are adversaries, so a lower loss does not always mean better images. Judge the model **visually** on photos it never saw during training, and with **FID** (next section).

In [ ]:
@torch.no_grad()
def stylize_batch(x):
    G_XY.eval()
    with torch.autocast(device.type, enabled=AMP):
        return G_XY(x.to(device)).float().cpu()

test_ds = ImageFolderList(test_photo_files, eval_tf)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=NUM_WORKERS)
idx = 0
for xb in test_loader:  # save every held-out result to results/
    for real, fake in zip(xb, stylize_batch(xb)):
        save_image(torch.cat([denorm(real), denorm(fake)], dim=2), RESULT_DIR / f"test_{idx:03d}.jpg")
        save_image(denorm(fake), RESULT_DIR / f"test_{idx:03d}_painting.png")
        idx += 1
print(f"Saved {idx} results to {RESULT_DIR}")

n_show = min(6, len(test_ds))
xs = torch.stack([test_ds[i] for i in range(n_show)])
fs = stylize_batch(xs)
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6.4), squeeze=False)
for i in range(n_show):
    axes[0, i].imshow(denorm(xs[i]).permute(1, 2, 0)); axes[1, i].imshow(denorm(fs[i]).permute(1, 2, 0))
    axes[0, i].axis("off"); axes[1, i].axis("off")
axes[0, 0].set_title("photo", loc="left"); axes[1, 0].set_title(ART_STYLE.replace("_", " "), loc="left")
plt.tight_layout(); fig.savefig(WORK_DIR / "gallery.jpg", dpi=110); plt.show()

In [ ]:
# Cycle consistency: photo -> painting -> reconstructed photo
@torch.no_grad()
def show_cycle(x):
    G_XY.eval(); G_YX.eval()
    fake = G_XY(x.unsqueeze(0).to(device)); rec = G_YX(fake)
    imgs, titles = [x, fake[0], rec[0]], ["photo", "photo → painting", "painting → photo (reconstruction)"]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
    for ax, im, t in zip(axes, imgs, titles):
        ax.imshow(denorm(im).permute(1, 2, 0)); ax.set_title(t); ax.axis("off")
    plt.tight_layout(); fig.savefig(WORK_DIR / "cycle.jpg", dpi=110); plt.show()
    print(f"reconstruction L1: {(denorm(rec[0]) - denorm(x)).abs().mean():.4f}")

show_cycle(test_ds[0])

## 10. Quantitative evaluation: FID
**Fréchet Inception Distance** between the stylized held-out photos and real paintings of the target style (lower is better). The baseline is FID(raw photos, paintings): if the model has learned the style, its FID should be clearly lower than the baseline. With only `N_TEST` = 64 images the absolute value is noisy and biased upward, so use it to **compare runs** (styles, epochs, λ values), not as an absolute number. Increase `N_TEST` for a tighter estimate.

FID here uses torchvision's ImageNet Inception-v3, so values are consistent across your runs but not directly comparable with published numbers, which use the TF-Inception port.

In [ ]:
import scipy.linalg

@torch.no_grad()
def inception_features(batches):
    net, feats = FID_NET.to(device), []
    for x in batches:
        x = nn.functional.interpolate(x.to(device).float(), size=(299, 299), mode="bilinear", align_corners=False)
        feats.append(net(x).double().cpu())
    return torch.cat(feats).numpy()

def frechet_distance(a, b):
    mu1, mu2 = a.mean(0), b.mean(0)
    s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    covmean = scipy.linalg.sqrtm(s1 @ s2)
    if not np.isfinite(covmean).all():  # near-singular covariances (few samples): add a small offset
        off = np.eye(len(s1)) * 1e-6
        covmean = scipy.linalg.sqrtm((s1 + off) @ (s2 + off))
    covmean = np.real(covmean)
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2 * np.trace(covmean))

if FID_NET is None:
    print("FID skipped: Inception weights not available (see the check cell near the top). Results above are unaffected.")
else:
    art_eval = ImageFolderList(random.Random(SEED).sample(art_files, min(len(art_files), 500)), eval_tf)
    load = lambda ds: DataLoader(ds, batch_size=32, num_workers=NUM_WORKERS)
    f_art = inception_features(load(art_eval))
    f_photo = inception_features(load(test_ds))
    f_fake = inception_features(stylize_batch(xb) for xb in load(test_ds))
    FID_NET.cpu()

    fid_model, fid_base = frechet_distance(f_fake, f_art), frechet_distance(f_photo, f_art)
    print(f"FID(stylized photos, {ART_STYLE}) = {fid_model:.1f}")
    print(f"FID(raw photos,      {ART_STYLE}) = {fid_base:.1f}  <- baseline (no translation)")
    json.dump({"style": ART_STYLE, "fid_model": fid_model, "fid_baseline": fid_base, "n_fake": len(test_ds),
               "n_real": len(art_eval), "backbone": "torchvision inception_v3 IMAGENET1K_V1",
               "epochs_trained": history[-1]["epoch"] if history else None},
              open(WORK_DIR / "metrics.json", "w"), indent=2)

## 11. Stylize your own photos
The generator is fully convolutional, so it can run at a **higher resolution than it was trained at**. Upload photos as a Kaggle dataset, set `MY_PHOTOS_DIR`, and run the cell below. Results go to `results/custom/`.

In [ ]:
@torch.no_grad()
def stylize_file(path, max_side=1024):
    img = safe_open(path)
    if img is None:
        return None
    s = min(1.0, max_side / max(img.size))
    w, h = (max(4, int(img.width * s) // 4 * 4), max(4, int(img.height * s) // 4 * 4))  # multiples of 4
    x = normalize(T.ToTensor()(img.resize((w, h), Image.BICUBIC))).unsqueeze(0)
    return T.ToPILImage()(denorm(stylize_batch(x)[0]))

MY_PHOTOS_DIR = None  # e.g. "/kaggle/input/my-photos"
if MY_PHOTOS_DIR:
    out = RESULT_DIR / "custom"; out.mkdir(exist_ok=True)
    for p in list_images(MY_PHOTOS_DIR):
        res = stylize_file(p)
        if res is not None:
            res.save(out / f"{p.stem}_{ART_STYLE}.jpg", quality=92)
    print("Saved to", out)
elif test_photo_files:
    demo = stylize_file(test_photo_files[0], max_side=IMG_SIZE * 2)
    plt.figure(figsize=(6, 6)); plt.imshow(demo); plt.axis("off"); plt.title(f"{demo.size[0]}x{demo.size[1]} inference"); plt.show()

## 12. Outputs
Everything is in `/kaggle/working`. After a *Save & Run All* commit you can download it from the **Output** tab:

| file | what |
|---|---|
| `generator_<style>.pt` | final photo→painting generator weights (~45 MB), all you need for inference |
| `checkpoints/last.pt` | full training state for resuming |
| `history.csv`, `losses.png` | per-epoch losses |
| `samples/epoch_XXX.jpg` | fixed-input preview after every epoch |
| `results/`, `gallery.jpg`, `cycle.jpg` | held-out results |
| `metrics.json` | FID score |

In [ ]:
for p in sorted(WORK_DIR.glob("*")):
    size = sum(f.stat().st_size for f in p.rglob("*")) if p.is_dir() else p.stat().st_size
    print(f"{size / 1e6:8.1f} MB  {p.name}")